In [ ]:
from z3 import *
import numpy as np, seaborn as sns, matplotlib.pyplot as plt

TARGET_NUMS=[0,1,16,23,37,88,138,272,449,528,750,1100]
ADD,MUL,DIV=0,1,2
SYM={ADD:"+",MUL:"*",DIV:"/"}
FINAL_N=55

REGIONS="""AAAAABBB
CCCDDEEB
CFCDDDEB
GFFHHIEE
GGFHHIIJ
GKFLIIMJ
GKLLLMMJ
KKKLMMJJ""".splitlines()

KNOWN_POS={15:(0,5),53:(0,7),9:(2,3),25:(2,5),39:(4,1),6:(4,4),46:(5,1),18:(5,3),32:(5,5),3:(5,6),0:(7,0)}

def checkpoint_moves(K):
    return [m for m in range(1,20) if m%3==1]+[19+j*K for j in range(1,6)]

def solve_path(K):
    n=IntVector("n",FINAL_N)
    op=IntVector("op",FINAL_N-1)
    s=Solver()
    s.add(n[0]==0)

    for i in range(1,FINAL_N):
        s.add(op[i-1]>=ADD,op[i-1]<=DIV,n[i-1]>=0)
        s.add(Or(
            And(op[i-1]==ADD,n[i]==n[i-1]+i),
            And(op[i-1]==MUL,n[i]==n[i-1]*i),
            And(op[i-1]==DIV,n[i-1]%i==0,n[i]*i==n[i-1])
        ))

    cps=[m-1 for m in checkpoint_moves(K)]
    for i in cps: s.add(Or([n[i]==x for x in TARGET_NUMS]))
    s.add(Distinct([n[i] for i in cps]))
    s.add(n[FINAL_N-1]==59400)

    if s.check()!=sat: return None
    M=s.model()
    scores=[M[n[i]].as_long() for i in range(FINAL_N)]
    ops=[M[op[i]].as_long() for i in range(FINAL_N-1)]
    return scores,ops,cps

def cell_is_region(r,c,region):
    return Or([And(r==rr,c==cc) for rr in range(8) for cc in range(8) if REGIONS[rr][cc]==region])

def solve_placement(scores):
    N=len(scores)
    ops=[]
    for i in range(1,N):
        a,b=scores[i-1],scores[i]
        ops.append(ADD if b==a+i else MUL if b==a*i else DIV)

    s=Solver()
    row=IntVector("row",N); col=IntVector("col",N); tower=BoolVector("tower",N)

    for i in range(N): s.add(row[i]>=0,row[i]<8,col[i]>=0,col[i]<8)
    s.add(Distinct([row[i]*8+col[i] for i in range(N)]))

    for i,(r,c) in KNOWN_POS.items(): s.add(row[i]==r,col[i]==c)
    s.add(tower[0]==True)

    for i in range(1,N):
        dr=row[i]-row[i-1]; dc=col[i]-col[i-1]
        knight=Or(And(Abs(dr)==2,Abs(dc)==1),And(Abs(dr)==1,Abs(dc)==2))
        height=Or(And(Abs(dr)==2,dc==0),And(dr==0,Abs(dc)==2))
        if ops[i-1]==ADD: s.add(tower[i]==tower[i-1],knight)
        else: s.add(tower[i]!=tower[i-1],height)

    for region in sorted(set("".join(REGIONS))):
        visits=[And(tower[i],cell_is_region(row[i],col[i],region)) for i in range(N)]
        s.add(PbEq([(x,1) for x in visits],1))

    if s.check()!=sat: return None,None
    M=s.model()
    path=[(M[row[i]].as_long(),M[col[i]].as_long(),is_true(M[tower[i]])) for i in range(N)]
    return path,ops

def build_grids(path,scores,ops):
    tour=np.full((8,8),np.nan); score=np.full((8,8),np.nan)
    towers=np.zeros((8,8)); tower_labels=np.full((8,8),"",dtype=object)
    op_grid=np.full((8,8),np.nan); op_labels=np.full((8,8),"",dtype=object)

    for i,(r,c,is_tower) in enumerate(path):
        tour[r,c]=i; score[r,c]=scores[i]
        if is_tower: towers[r,c]=1; tower_labels[r,c]=REGIONS[r][c]
        if i<len(ops): op_grid[r,c]=ops[i]; op_labels[r,c]=f"{SYM[ops[i]]}{i+1}"
    return tour,score,towers,tower_labels,op_grid,op_labels

def heat(data,title,labels=None,fmt=".0f",mask=None):
    plt.figure(figsize=(3,3))
    sns.heatmap(data,annot=labels if labels is not None else True,fmt=fmt,
                linewidths=.5,square=True,mask=mask,cbar=False,cmap="tab20")
    plt.axis("off"); plt.title(title); plt.tight_layout(); plt.show()

def answer(score):
    total=0
    for r in range(8):
        for c in range(8):
            if np.isnan(score[r,c]):
                for dr,dc in [(1,0),(-1,0),(0,1),(0,-1)]:
                    rr,cc=r+dr,c+dc
                    if 0<=rr<8 and 0<=cc<8 and not np.isnan(score[rr,cc]):
                        total+=int(score[rr,cc])
    return total

for K in range(1,10):
    solved=solve_path(K)
    if solved is None: continue
    scores,ops_from_path,cps=solved
    path,ops=solve_placement(scores)
    if path is None: continue

    print("K =",K)
    print("Checkpoint scores:",[(i,scores[i]) for i in cps])
    print("Operations:",[SYM[x] for x in ops])

    tour,score,towers,tower_labels,op_grid,op_labels=build_grids(path,scores,ops)
    heat(tour,"Tour Indices",mask=np.isnan(tour))
    heat(score,"Scores",mask=np.isnan(score))
    heat(towers,"Tower Regions",labels=tower_labels,fmt="")
    heat(op_grid,"Operations",labels=op_labels,fmt="",mask=np.isnan(op_grid))
    print("Answer:",answer(score))
    break